# Data Cleaning & Customer Incremental Dataset Generation

## Step 1: Load Raw Superstore Dataset & Extract Customer Attributes

In [1]:
import pandas as pd
import numpy as np

# Load Raw Superstore Dataset
df_superstore = pd.read_csv('../data/superstore.csv', encoding='latin1')
print('Raw Superstore Orders Shape:', df_superstore.shape)

# Extract Customer Dimension Fields
cust_cols = ['Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region']
df_cust_raw = df_superstore[cust_cols].copy()
df_cust_raw.columns = [c.lower().replace(' ', '_') for c in df_cust_raw.columns]

# Introduce test null values
df_cust_raw.loc[df_cust_raw['customer_id'] == 'CG-12520', 'postal_code'] = np.nan
df_cust_raw.loc[df_cust_raw['customer_id'] == 'DV-13045', 'region'] = np.nan

Raw Superstore Orders Shape: (9994, 21)


## Step 2: Inspect Null Counts BEFORE & AFTER Cleaning

In [2]:
print('--- Null Counts BEFORE Cleaning ---')
print(df_cust_raw.isnull().sum())

# Fill Nulls & Deduplicate Order Transactions into Unique Customers
df_cust_clean = df_cust_raw.copy()
df_cust_clean['postal_code'] = df_cust_clean['postal_code'].fillna('00000').astype(str)
df_cust_clean['region'] = df_cust_clean['region'].fillna('Unknown')
df_master_clean = df_cust_clean.drop_duplicates(subset=['customer_id']).sort_values(by='customer_id').reset_index(drop=True)

print('\nRaw Superstore Order Rows:      ', len(df_superstore))
print('Extracted Customer Order Rows:', len(df_cust_raw))
print('Clean Unique Customer Master: ', len(df_master_clean))

print('\n--- Null Counts AFTER Cleaning ---')
print(df_master_clean.isnull().sum())

# Save Clean Baseline File
df_master_clean.head(100).to_csv('../data/customer_master.csv', index=False)
print('\nSaved cleaned master dataset to data/customer_master.csv')

--- Null Counts BEFORE Cleaning ---
customer_id      0
customer_name    0
segment          0
country          0
city             0
state            0
postal_code      5
region           9
dtype: int64

Raw Superstore Order Rows:       9994
Extracted Customer Order Rows: 9994
Clean Unique Customer Master:  793

--- Null Counts AFTER Cleaning ---
customer_id      0
customer_name    0
segment          0
country          0
city             0
state            0
postal_code      0
region           0
dtype: int64

Saved cleaned master dataset to data/customer_master.csv


## Step 3: Generate Customer Incremental Dataset (customer_incremental.csv)

This step creates `data/customer_incremental.csv` simulating incoming batch data:
- **10 Updates to Existing Customers**: Modifies `city`, `state`, and `segment` for 10 real Superstore customers.
- **5 New Customer Inserts**: Adds 5 brand new customer records (`NEW-001` through `NEW-005`).


In [3]:
# 10 Updates to existing customers
inc_updates = df_master_clean.head(10).copy()
inc_updates['city'] = ['New York', 'Los Angeles', 'Chicago', 'Houston', 'Phoenix', 'Philadelphia', 'San Antonio', 'San Diego', 'Dallas', 'San Jose']
inc_updates['state'] = ['New York', 'California', 'Illinois', 'Texas', 'Arizona', 'Pennsylvania', 'Texas', 'California', 'Texas', 'California']
inc_updates['segment'] = ['Corporate', 'Home Office', 'Consumer', 'Corporate', 'Home Office', 'Consumer', 'Corporate', 'Home Office', 'Consumer', 'Corporate']

# 5 New Customer Inserts
new_custs = pd.DataFrame([
    {'customer_id': 'NEW-001', 'customer_name': 'Alex Turner', 'segment': 'Consumer', 'country': 'United States', 'city': 'Seattle', 'state': 'Washington', 'postal_code': '98101', 'region': 'West'},
    {'customer_id': 'NEW-002', 'customer_name': 'Bianca Dev', 'segment': 'Corporate', 'country': 'United States', 'city': 'Boston', 'state': 'Massachusetts', 'postal_code': '02108', 'region': 'East'},
    {'customer_id': 'NEW-003', 'customer_name': 'Charlie Hayes', 'segment': 'Home Office', 'country': 'United States', 'city': 'Austin', 'state': 'Texas', 'postal_code': '78701', 'region': 'Central'},
    {'customer_id': 'NEW-004', 'customer_name': 'Diana Prince', 'segment': 'Consumer', 'country': 'United States', 'city': 'Denver', 'state': 'Colorado', 'postal_code': '80202', 'region': 'West'},
    {'customer_id': 'NEW-005', 'customer_name': 'Ethan Hunt', 'segment': 'Corporate', 'country': 'United States', 'city': 'Miami', 'state': 'Florida', 'postal_code': '33101', 'region': 'South'}
])

inc_df = pd.concat([inc_updates, new_custs], ignore_index=True)
inc_df.to_csv('../data/customer_incremental.csv', index=False)
print('Generated customer_incremental.csv successfully! Total Rows:', len(inc_df))
inc_df

Generated customer_incremental.csv successfully! Total Rows: 15


,customer_id,customer_name,segment,country,city,state,postal_code,region
0,AA-10315,Alex Avila,Corporate,United States,New York,New York,55407.0,Central
1,AA-10375,Allen Armold,Home Office,United States,Los Angeles,California,85204.0,West
2,AA-10480,Andrew Allen,Consumer,United States,Chicago,Illinois,28027.0,South
3,AA-10645,Anna Andreadi,Corporate,United States,Houston,Texas,19013.0,East
4,AB-10015,Aaron Bergman,Home Office,United States,Phoenix,Arizona,98103.0,West
5,AB-10060,Adam Bellavance,Consumer,United States,Philadelphia,Pennsylvania,10009.0,East
6,AB-10105,Adrian Barton,Corporate,United States,San Antonio,Texas,85023.0,West
7,AB-10150,Aimee Bixby,Home Office,United States,San Diego,California,11561.0,East
8,AB-10165,Alan Barnes,Consumer,United States,Dallas,Texas,90036.0,West
9,AB-10255,Alejandro Ballentine,Corporate,United States,San Jose,California,44052.0,East
